In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('bitcoin_clustered_users.csv')

# Изолируем только финансовое ядро (Кластер №3 "Элита / Биржи")
df_elite = df[df['Cluster'] == 3].copy()

# Подготовка признаков (логарифмирование + масштабирование)
# Берем объемы и количество транзакций
features = df_elite[['Total_Received_BTC', 'Total_Tx']].copy()
features['Total_Received_BTC'] = np.log1p(features['Total_Received_BTC'])
features['Total_Tx'] = np.log1p(features['Total_Tx'])

# Z-нормализация
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)

# DBSCAN (микро-сегментация)
dbscan = DBSCAN(eps=0.5, min_samples=10)
df_elite['SubCluster'] = dbscan.fit_predict(X_scaled)

print("--- КОЛИЧЕСТВО СУЩНОСТЕЙ В ПОДКЛАСТЕРАХ ---")
print(df_elite['SubCluster'].value_counts())
print("\n")

# СЧИТАЕМ ДИСПЕРСИЮ (Стандартное отклонение)
std_devs = df_elite.groupby('SubCluster')[['Total_Received_BTC', 'Total_Tx']].std()

print("--- СТАНДАРТНОЕ ОТКЛОНЕНИЕ (Std) ---")
print(std_devs)

--- КОЛИЧЕСТВО СУЩНОСТЕЙ В ПОДКЛАСТЕРАХ ---
SubCluster
 0    9533
-1      22
Name: count, dtype: int64


--- СТАНДАРТНОЕ ОТКЛОНЕНИЕ (Std) ---
            Total_Received_BTC       Total_Tx
SubCluster                                   
-1               473411.349365  496375.045311
 0                 6154.520048    1176.186758


In [2]:
#  Делаем радиус ОЧЕНЬ маленьким (было 0.5, ставим 0.01)
#  Уменьшаем минимальное кол-во соседей до 5 (чтобы поймать даже маленькие группы ботов)
dbscan = DBSCAN(eps=0.01, min_samples=5) 
df_elite['SubCluster'] = dbscan.fit_predict(X_scaled)

print("--- НОВЫЕ ПОДКЛАСТЕРЫ (строгий поиск) ---")
print(df_elite['SubCluster'].value_counts())

print("\n--- НОВАЯ ДИСПЕРСИЯ (Std) ---")
print(df_elite.groupby('SubCluster')[['Total_Received_BTC', 'Total_Tx']].std())

--- НОВЫЕ ПОДКЛАСТЕРЫ (строгий поиск) ---
SubCluster
-1     8826
 7       59
 20      49
 5       48
 26      46
       ... 
 64       5
 63       5
 59       5
 25       4
 41       4
Name: count, Length: 68, dtype: int64

--- НОВАЯ ДИСПЕРСИЯ (Std) ---
            Total_Received_BTC      Total_Tx
SubCluster                                  
-1                27085.760151  25520.636605
 0                    0.012335      0.000000
 1                    0.017502      6.436748
 2                    0.012982      0.000000
 3                    0.006411      0.000000
...                        ...           ...
 62                   0.018359      0.000000
 63                   0.005386      0.000000
 64                   0.342330      0.547723
 65                   0.013076      4.898979
 66                   0.014812      0.000000

[68 rows x 2 columns]
